# Gold — dim_commenter_scd1 (SCD 1, poc method)

One row per comment author.
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

Natural key `commenter_nk = COALESCE(author_channel_id, 'name:' || author_display_name)` (stored, it is the MERGE key); `commenter_key = xxhash64(commenter_nk)` as in `dim_commenter`.

**Changed keys** = authors with a comment read in the window (`stg_comments.snapshot_date`); their `first_comment_at` / `last_comment_at` / latest name are recomputed over **all** their comments (`stg_comments_latest`). No `modified_at`, so no newer-row guard (`order_col=None`, poc behaviour).

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
NK = F.coalesce(F.col("author_channel_id"), F.concat(F.lit("name:"), F.col("author_display_name")))
comments = spark.table(f"{SILVER}.stg_comments_latest").withColumn("commenter_nk", NK).where("commenter_nk IS NOT NULL")
keys = None if FULL else changed_keys(
    spark.table(f"{SILVER}.stg_comments").where(in_window(F.col("snapshot_date"), FROM_DATE, TO_DATE))
         .select(NK.alias("commenter_nk")))
if keys is not None:
    comments = comments.join(keys, "commenter_nk", "left_semi")

from pyspark.sql.window import Window
last = Window.partitionBy("commenter_nk").orderBy(F.col("published_at").desc(), F.col("comment_id"))
src = (comments
       .withColumn("rn", F.row_number().over(last))
       .withColumn("first_comment_at", F.min("published_at").over(Window.partitionBy("commenter_nk")))
       .withColumn("last_comment_at", F.max("published_at").over(Window.partitionBy("commenter_nk")))
       .where("rn = 1")
       .select(F.xxhash64("commenter_nk").alias("commenter_key"), "commenter_nk", "author_channel_id",
               "author_display_name", "first_comment_at", "last_comment_at")
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["commenter_nk"], exclude=["commenter_key"])

In [ ]:
load_scd1(src, f"{GOLD}.dim_commenter_scd1", """
        commenter_key BIGINT, commenter_nk STRING, author_channel_id STRING, author_display_name STRING,
        first_comment_at TIMESTAMP, last_comment_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Comment author dimension, SCD 1 (poc checksum method). commenter_key = xxhash64(commenter_nk), -1 = Unknown.",
    ["commenter_nk"], LOAD_TYPE, {"commenter_key": "-1", "author_display_name": "'Unknown'",
                                  "etl_date": "current_timestamp()", "etl_by": "'Databricks'"}, order_col=None)